# Soybean leaf-disease recognition: reproduction from the project zip in Google Drive (Colab, T4 GPU)



## 0. Setup

In [1]:
# Mount Drive, find the project there, unpack it to /content/project and cd into it. No GitHub needed.
from google.colab import drive
drive.mount('/content/drive')

import glob, hashlib, os, shutil, zipfile
from pathlib import Path

SOURCE  = ''     # empty = auto-find AI-Farmer-Advisory-Project*.zip (or an unpacked AI-Farmer-Advisory-Project folder) in My Drive.
                 # Or set the exact path, e.g. '/content/drive/MyDrive/projects/AI-Farmer-Advisory-Project_leaf_only.zip'
PROJECT = '/content/project'

def find_sources(drive_root='/content/drive/MyDrive'):
    """Zips / folders named AI-Farmer-Advisory-Project* in My Drive and up to two levels below it (a full Drive scan is slow)."""
    found = []
    for depth in range(3):
        found += glob.glob(os.path.join(drive_root, *['*'] * depth, 'AI-Farmer-Advisory-Project*'))
    ok = [p for p in found if (p.endswith('.zip') and os.path.isfile(p))
          or (os.path.isdir(p) and os.path.exists(os.path.join(p, 'soya_pipeline', 'reproduce.py')))]
    return sorted(set(ok), key=lambda p: ('leaf_only' not in os.path.basename(p), -os.path.getmtime(p)))   # leaf_only first, then newest

if SOURCE:
    src = SOURCE
else:
    cands = find_sources()
    print('found in Drive:', *cands, sep='\n  ')
    assert cands, ('No AI-Farmer-Advisory-Project*.zip (or folder) found in My Drive. '
                   'Upload the project zip to Drive, or set SOURCE to its exact path.')
    src = cands[0]
print('using:', src)

tmp = '/content/_unpack'
shutil.rmtree(tmp, ignore_errors=True)
if os.path.isdir(src):
    shutil.copytree(src, tmp, ignore=shutil.ignore_patterns('artifacts', 'data', '.git', '__pycache__', '*.pt', '*.npy'))
    source_id = f'folder {src}'
else:
    with zipfile.ZipFile(src) as z:
        z.extractall(tmp)
    source_id = f'{os.path.basename(src)} md5={hashlib.md5(open(src, "rb").read()).hexdigest()}'
roots = [p.parent.parent for p in Path(tmp).rglob('reproduce.py') if p.parent.name == 'soya_pipeline']
assert roots, 'soya_pipeline/reproduce.py not found in the source - is this the right zip?'
shutil.copytree(roots[0], PROJECT, dirs_exist_ok=True)     # merge: results in soya_pipeline/artifacts survive a re-run of this cell
shutil.rmtree(tmp, ignore_errors=True)
os.environ['SOYA_SOURCE'] = source_id                      # recorded in every reproduction report
%cd /content/project/soya_pipeline
print('source:', source_id)
!nvidia-smi -L
assert os.path.exists('requirements-core.txt'), 'requirements-core.txt is missing: this is not the current project version.'

Mounted at /content/drive
found in Drive:
  /content/drive/MyDrive/Colab Notebooks/AI-Farmer-Advisory-Project_leaf_only/AI-Farmer-Advisory-Project
using: /content/drive/MyDrive/Colab Notebooks/AI-Farmer-Advisory-Project_leaf_only/AI-Farmer-Advisory-Project
/content/project/soya_pipeline
source: folder /content/drive/MyDrive/Colab Notebooks/AI-Farmer-Advisory-Project_leaf_only/AI-Farmer-Advisory-Project
GPU 0: Tesla T4 (UUID: GPU-bbe5457e-cca0-a760-b91e-3d2123dc21ba)


In [2]:
# Colab already ships PyTorch / torchvision; this only adds what is missing and checks the minimum versions.
!pip install -q -r requirements.txt pytest


In [3]:
# Back results up to Google Drive after every step, so a disconnected run can be resumed.
# Set SYNC = '' to disable. (A Colab disconnect wipes /content; Drive survives.)
from google.colab import drive
drive.mount('/content/drive')              # already mounted by the setup cell: no-op
SYNC  = '/content/drive/MyDrive/soya_artifacts'
FLAGS = f"--resume --sync-dir {SYNC}" if SYNC else "--resume"
print('reproduce.py flags:', FLAGS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
reproduce.py flags: --resume --sync-dir /content/drive/MyDrive/soya_artifacts


### Dataset (downloaded once from Mendeley Data; please respect the dataset licence and cite it)
The Mendeley archive contains the leaf photos **and** the UAV frames. Only the leaf folder is used here, so UAV archives are left unextracted.


In [4]:
import glob, os, zipfile

DATA = '/content/data'
URL  = "https://data.mendeley.com/public-api/zip/hkbgh5s3b7/download/1"

def unzip_all(root):
    """Extract every .zip under `root` (including zips inside zips) next to itself, then delete it.
    The UAV half of the dataset is left alone: a zip is skipped when its own file name mentions UAV or when it sits inside the folder
    `Soyabean_UAV-Based_Image_Dataset`. (The archive's top folder is also called "...UAV and Leaf...", so the whole path must NOT be
    tested: that skipped the leaf zips too.)"""
    def is_uav(z):
        parts = os.path.relpath(z, root).lower().split(os.sep)
        return 'uav' in parts[-1] or any('uav-based' in p for p in parts[:-1])
    found = True
    while found:
        found = False
        for z in glob.glob(f'{root}/**/*.zip', recursive=True):
            if is_uav(z):
                continue
            with zipfile.ZipFile(z) as f:
                f.extractall(os.path.dirname(z))
            os.remove(z); found = True

if not glob.glob(f'{DATA}/**/Soyabean_Leaf_Image_Dataset/**/*.jpg', recursive=True):
    os.makedirs(DATA, exist_ok=True)
    !wget -q -O {DATA}/soya.zip "{URL}"
    unzip_all(DATA)

leaf = glob.glob(f'{DATA}/**/Soyabean_Leaf_Image_Dataset', recursive=True)
assert leaf, 'Soyabean_Leaf_Image_Dataset not found - check the download'
os.environ['SOYA_ROOT'] = os.path.dirname(leaf[0])
n_img = len(glob.glob(f'{leaf[0]}/**/*.jpg', recursive=True))
print('SOYA_ROOT =', os.environ['SOYA_ROOT'], '|', n_img, 'jpg files in the leaf folder')

SOYA_ROOT = /content/data/MH-SoyaHealthVision An Indian UAV and Leaf Image Dataset for Integrated Crop Health Assessment | 2782 jpg files in the leaf folder


### Environment record and test-suite

In [5]:
# Library versions of this run. Keep the output as soya_pipeline/requirements-lock.txt in the project (and the next zip) if it differs from the zip's.
!pip freeze | grep -iE "^(numpy|pandas|scikit-learn|scikit-image|opencv-python-headless|pillow|joblib|matplotlib|torch|torchvision)==" | tee /content/requirements-lock.txt

joblib==1.6.0
matplotlib==3.10.0
numpy==2.1.3
opencv-python-headless==5.0.0.93
pandas==2.2.3
pillow==11.3.0
scikit-image==0.25.2
scikit-learn==1.6.1
torch==2.11.0+cu130
torchvision==0.26.0+cu130


In [6]:
# 24 tests on synthetic data (about 2 min on Colab CPU). All must pass before the long runs.
!python -m pytest


........................                                                 [100%]
24 passed in 191.13s (0:03:11)


### PyTorch smoke test (about 2 minutes, synthetic data)
The unit tests above cannot run PyTorch code on a CPU-only runner, so this cell runs the GPU path end to end on a tiny synthetic dataset:
deep features, fine-tuning (2 epochs), CNN evaluation with TTA, an explicit 2-model ensemble, and prediction on one image.
**If any line fails, stop and fix it here instead of 60 minutes into the real run.**

In [7]:
%%bash
set -eo pipefail          # a failure inside a `| tail` pipe must stop the cell too
python make_dummy_data.py --out /content/dummy --per-class 60 > /dev/null
export SOYA_ROOT=/content/dummy
A="--dataset leaf --img-size 64 --artifacts /content/smoke"
python 01_build_manifest.py $A --folds 3 > /dev/null
python 02_cache_images.py $A | tail -1
python 04_extract_features.py $A | tail -2                      # hand + frozen MobileNetV3 features (needs PyTorch)
python 06_train_classical.py $A --fold 0 | tail -1
python 07_finetune_cnn.py $A --fold 0 --epochs 2 --warmup-epochs 1 --batch-size 16 | tail -2
python 07_finetune_cnn.py $A --fold 0 --epochs 2 --warmup-epochs 1 --batch-size 16 --cutmix 1.0 | tail -1
python 08_evaluate.py $A --fold 0 --which cnn --cnn mobilenet_v3_large_64,mobilenet_v3_large_64_cutmix --tta --bootstrap 20 | grep -E '"model"|"macro_f1"'
python 10_predict.py --dataset leaf --artifacts /content/smoke --input /content/dummy/Soyabean_Leaf_Image_Dataset/Healthy/img_000.jpg --ckpt "/content/smoke/leaf/models/fold0/cnn_*.pt"
echo "PYTORCH SMOKE TEST PASSED"

(366, 64, 64, 3) -> /content/smoke/leaf/cache/all_x64.npy (4 MB)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth
deep: (366, 960) (mobilenet_v3_large, device=cuda)
Best on VAL: hand + svc (macro-F1=0.8527)

Best val macro-F1 = 0.3553 -> /content/smoke/leaf/models/fold0/cnn_mobilenet_v3_large_64.pt  (test set untouched)
Best val macro-F1 = 0.5078 -> /content/smoke/leaf/models/fold0/cnn_mobilenet_v3_large_64_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.4641,
img_000.jpg: Healthy 40.6%, Rust 26.5%, Pest_Attack 13.2%
PYTORCH SMOKE TEST PASSED


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 372 out of 372 | elapsed:    1.4s finished
100%|██████████| 21.1M/21.1M [00:00<00:00, 87.6MB/s]


## 1. Leaf (6 classes) - 5-fold cross-validation
`reproduce.py` runs: manifest + folds, cache, EDA, features, audit, then per fold the classical models and the CNNs, and finally the
pooled out-of-fold summaries. Pick one recipe:

* `leaf_ensemble3`: EfficientNet-B0 + EfficientNet-B0/CutMix + ConvNeXt-Tiny (reported macro-F1 0.761)
* `leaf_baseline`: EfficientNet-B0 only (reported macro-F1 0.733)

Preview the exact commands first with `!python reproduce.py {RECIPE} --dry-run`.


In [8]:
RECIPE = 'leaf_ensemble3'      # or 'leaf_baseline'
!python reproduce.py {RECIPE} {FLAGS}



[1/55] 01_build_manifest.py --dataset leaf --img-size 320 --artifacts /content/project/soya_pipeline/artifacts --folds 5
Found 2782 images in /content/data/MH-SoyaHealthVision An Indian UAV and Leaf Image Dataset for Integrated Crop Health Assessment/Soyabean_Leaf_Image_Dataset
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 740 tasks      | elapsed:   33.2s
[Parallel(n_jobs=-1)]: Done 2780 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 2782 out of 2782 | elapsed:  2.0min finished

Classes (VERIFY these are your disease / pest / healthy folders):
                                        images  source_groups  clusters  largest_cluster
label                                                                                   
Caterpillar and Semilooper Pest Attack     577            577       577                1
Healthy_Soyabean                           204            204       204                1
Soyabean_Frog_Leaf_Eye 

In [9]:
import json, pandas as pd
rep = json.load(open(f'artifacts/leaf/reports/cv/reproduction_{RECIPE}.json'))
print('GPU:', rep['environment']['gpu'], '| torch', rep['environment']['packages']['torch'], '| commit', rep['environment']['git_commit'], '| minutes', rep['minutes'], '| steps skipped by --resume:', rep['steps_skipped_by_resume'])
pd.DataFrame([{'model': t, 'macro_f1': r['pooled_macro_f1'], 'ci_low': r['pooled_macro_f1_ci95'][0], 'ci_high': r['pooled_macro_f1_ci95'][1],
               'accuracy': r['pooled_accuracy'], 'folds': list(r['per_fold_macro_f1'].values())} for t, r in rep['results'].items()]).round(4)

GPU: Tesla T4 | torch 2.11.0+cu130 | commit None | minutes 110.9 | steps skipped by --resume: 0


,model,macro_f1,ci_low,ci_high,accuracy,folds
0,classical,0.6904,0.6709,0.7085,0.7242,"[0.6709, 0.6983, 0.7004, 0.6972, 0.6822]"
1,cnn_efficientnet_b0_320_tta,0.7340,0.7183,0.7505,0.7485,"[0.7206, 0.7407, 0.7424, 0.7413, 0.7221]"
2,cnn_efficientnet_b0_320_cutmix_tta,0.7290,0.7124,0.7462,0.7395,"[0.7389, 0.7399, 0.7114, 0.7388, 0.7143]"
3,cnn_convnext_tiny_320_tta,0.7511,0.7344,0.7687,0.7656,"[0.7483, 0.7467, 0.7909, 0.7416, 0.7244]"
4,cnn_ensemble3_tta,0.7602,0.7438,0.7770,0.7747,"[0.7647, 0.7675, 0.7552, 0.7642, 0.7456]"


## 2. Optional: DINOv2 frozen features (negative result: 0.684 alone; blend 0.749)
Runs after section 1 (needs the leaf cache, folds and baseline predictions). Cells are the exact code that produced the numbers in `docs/RESULTS.md`.

In [10]:
import torch, numpy as np, torch.nn.functional as F
NAME, SIZE = 'dinov2_vits14', 224      # SIZE must be divisible by 14
FEAT_FILE = f'artifacts/leaf/features/dino_{NAME}_{SIZE}.npy'
OOF_FILE  = f'artifacts/leaf/reports/oof_{NAME}_{SIZE}.csv'

X = np.load('artifacts/leaf/cache/all_x320.npy', mmap_mode='r')
dev = 'cuda'
net = torch.hub.load('facebookresearch/dinov2', NAME).to(dev).eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
out = []
with torch.no_grad(), torch.autocast('cuda'):
    for i in range(0, len(X), 64):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + 64])).to(dev).permute(0, 3, 1, 2).float() / 255
        xb = F.interpolate(xb, size=(SIZE, SIZE), mode='bilinear', antialias=True, align_corners=False)
        out.append(net((xb - mean) / std).float().cpu())
E = torch.cat(out).numpy(); np.save(FEAT_FILE, E); print(E.shape)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 272MB/s]
/tmp/ipykernel_2425/1882315970.py:14: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  xb = torch.from_numpy(np.ascontiguousarray(X[i:i + 64])).to(dev).permute(0, 3, 1, 2).float() / 255


(2756, 384)


In [11]:
import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import f1_score
from metrics import cluster_bootstrap_f1

E = np.load(FEAT_FILE)
m = pd.read_csv('artifacts/leaf/manifest.csv')
classes = sorted(m.label.unique()); y = m.label_id.to_numpy(); K = len(classes)
P = np.zeros((len(m), K)); fold_f1 = []
for f in range(m.fold.max() + 1):
    te, tr = (m.fold == f).to_numpy(), (m.fold != f).to_numpy()
    gs = GridSearchCV(make_pipeline(StandardScaler(), SVC(class_weight='balanced')), {'svc__C': [1, 10, 100]},
                      cv=StratifiedGroupKFold(3, shuffle=True, random_state=0), scoring='f1_macro', n_jobs=-1)
    gs.fit(E[tr], y[tr], groups=m.cluster[tr])
    C = gs.best_params_['svc__C']
    clf = make_pipeline(StandardScaler(), SVC(C=C, class_weight='balanced', probability=True, random_state=0)).fit(E[tr], y[tr])
    P[te] = clf.predict_proba(E[te])
    fold_f1.append(f1_score(y[te], P[te].argmax(1), average='macro'))
    print(f'fold {f}: C={C}  test macro-F1 {fold_f1[-1]:.4f}')
pred = P.argmax(1)
lo, hi = cluster_bootstrap_f1(y, pred, m.cluster.to_numpy(), 1000, 0)
print(f'\nPOOLED macro-F1 {f1_score(y, pred, average="macro"):.4f}  (95% CI {lo:.3f}-{hi:.3f})  | folds mean {np.mean(fold_f1):.4f}')
pd.DataFrame(P, columns=[f'p_{c}' for c in classes]).assign(relpath=m.relpath, fold=m.fold, label_id=y).to_csv(OOF_FILE, index=False)

fold 0: C=1  test macro-F1 0.6556
fold 1: C=1  test macro-F1 0.6981
fold 2: C=1  test macro-F1 0.6982
fold 3: C=1  test macro-F1 0.6856
fold 4: C=1  test macro-F1 0.6806

POOLED macro-F1 0.6837  (95% CI 0.665-0.702)  | folds mean 0.6836


In [12]:
CNN_TAG = 'cnn_efficientnet_b0_320_tta'      # or 'cnn_ensemble3_tta'
from sklearn.metrics import f1_score
oof = pd.read_csv(OOF_FILE).set_index('relpath'); pc = [c for c in oof.columns if c.startswith('p_')]
parts = []
for f in range(m.fold.max() + 1):
    c = pd.read_csv(f'artifacts/leaf/reports/fold{f}/{CNN_TAG}_pred.csv'); d = oof.loc[c.relpath]
    parts.append(pd.DataFrame({'y': c.label_id.values, 'cluster': c.cluster.values, 'cnn': list(c[pc].to_numpy()), 'dino': list(d[pc].to_numpy())}))
d = pd.concat(parts, ignore_index=True)
y2, cl = d.y.to_numpy(), d.cluster.to_numpy(); A, B = np.stack(d.cnn), np.stack(d.dino)
for name, p in [('CNN only', A), ('DINOv2 only', B), ('50/50 blend', 0.5 * A + 0.5 * B)]:
    pr = p.argmax(1); lo, hi = cluster_bootstrap_f1(y2, pr, cl, 1000, 0)
    print(f'{name:12s} pooled macro-F1 {f1_score(y2, pr, average="macro"):.4f}  (95% CI {lo:.3f}-{hi:.3f})')

CNN only     pooled macro-F1 0.7340  (95% CI 0.716-0.751)
DINOv2 only  pooled macro-F1 0.6837  (95% CI 0.665-0.702)
50/50 blend  pooled macro-F1 0.7514  (95% CI 0.735-0.768)


## 3. Keep the results (Colab deletes the runtime on reset)

In [13]:
!mkdir -p /content/drive/MyDrive/soya_artifacts
!cp /content/requirements-lock.txt /content/drive/MyDrive/soya_artifacts/ 2>/dev/null || true
!rsync -a --exclude cache artifacts/ /content/drive/MyDrive/soya_artifacts/ 2>/dev/null || true
!ls /content/drive/MyDrive/soya_artifacts

leaf  requirements-lock.txt


## 4. After a successful run
1. Add one row per recipe to the log in `docs/RESULTS.md` (date, **source zip md5** printed by the setup cell, GPU / torch, pooled macro-F1, CI, difference to the
   reported value, tests passed). Differences up to about 0.01 macro-F1 are expected from GPU non-determinism; larger ones should be investigated.
2. If `/content/requirements-lock.txt` differs from the one in the zip, add it as `soya_pipeline/requirements-lock.txt` and make a new zip.
3. Save the notebook **without outputs** (Edit > Clear all outputs) before putting it back into the zip.